

##Prepare Environment



In [1]:
!pip install --upgrade pip
!pip install --upgrade datasets[audio] transformers accelerate evaluate jiwer tensorboard gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 38.5 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 7.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 20.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 29.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 24.6 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 37.1 MB/s  0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 1.14.0
    Uninstalling gradio_client-1.14.0:
      Successfully uninstalled gradio_client-1.14.0
  Attempting uninstall: datasets
    Found existing install

In [2]:
!pip install -U transformers datasets

## Login to HF

In [3]:
from huggingface_hub import notebook_login

notebook_login()

## Load Dataset

In [4]:
!apt-get install -y jq

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
jq is already the newest version (1.6-2.1ubuntu3.1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.


In [5]:
!RESPONSE=$(curl -X POST "https://mozilladatacollective.com/api/datasets/cmn2cxzy701iumm077t5ayw0e/download" \
  -H "Authorization: Bearer API-KEY" \
  -H "Content-Type: application/json") && \
DOWNLOAD_URL=$(echo $RESPONSE | jq -r '.downloadUrl') && \
curl -L -o "hindi_cv.tar.gz" "$DOWNLOAD_URL"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   890    0   890    0     0   1084      0 --:--:-- --:--:-- --:--:--  1084
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  544M  100  544M    0     0  60.7M      0  0:00:08  0:00:08 --:--:-- 72.1M


## Extract dataset

In [6]:
!tar -xvzf hindi_cv.tar.gz

Streaming output truncated to the last 5000 lines.
cv-corpus-25.0-2026-03-09/hi/clips/common_voice_hi_37579833.mp3
cv-corpus-25.0-2026-03-09/hi/clips/common_voice_hi_37579835.mp3
cv-corpus-25.0-2026-03-09/hi/clips/common_voice_hi_37579836.mp3
cv-corpus-25.0-2026-03-09/hi/clips/common_voice_hi_37579838.mp3
cv-corpus-25.0-2026-03-09/hi/clips/common_voice_hi_37579839.mp3
cv-corpus-25.0-2026-03-09/hi/clips/common_voice_hi_37579843.mp3
cv-corpus-25.0-2026-03-09/hi/clips/common_voice_hi_37579844.mp3
cv-corpus-25.0-2026-03-09/hi/clips/common_voice_hi_37579845.mp3
cv-corpus-25.0-2026-03-09/hi/clips/common_voice_hi_37579846.mp3
cv-corpus-25.0-2026-03-09/hi/clips/common_voice_hi_37579847.mp3
cv-corpus-25.0-2026-03-09/hi/clips/common_voice_hi_37579853.mp3
cv-corpus-25.0-2026-03-09/hi/clips/common_voice_hi_37579854.mp3
cv-corpus-25.0-2026-03-09/hi/clips/common_voice_hi_37579855.mp3
cv-corpus-25.0-2026-03-09/hi/clips/common_voice_hi_37579856.mp3
cv-corpus-25.0-2026-03-09/hi/clips/common_voice_hi_37

## Load it into Python

In [7]:
import pandas as pd
import os

data_dir = "/content/cv-corpus-25.0-2026-03-09/hi"

train_df = pd.read_csv(f"{data_dir}/train.tsv", sep="\t")
test_df = pd.read_csv(f"{data_dir}/test.tsv", sep="\t")

# Prepend the full path to the audio files
train_df["path"] = data_dir + "/clips/" + train_df["path"]
test_df["path"] = data_dir + "/clips/" + test_df["path"]

train_df.head()

,client_id,path,sentence_id,sentence,sentence_domain,up_votes,down_votes,age,gender,accents,variant,locale,segment
0,0f018a99663f33afbb7d38aee281fb1afcfd07f9e7acd0...,/content/cv-corpus-25.0-2026-03-09/hi/clips/co...,6cd05fc0979a495e7008b614e213c3cac9926dae51ac76...,हमने उसका जन्मदिन मनाया।,NaN,2,0,NaN,NaN,NaN,NaN,hi,NaN
1,0f018a99663f33afbb7d38aee281fb1afcfd07f9e7acd0...,/content/cv-corpus-25.0-2026-03-09/hi/clips/co...,76bb7b27759e9acf98f9709f962731ad21362a9a8278be...,"साउथ दिल्ली नगर निगम सख्त, शॉपिंग मॉल के बाहर ...",NaN,2,0,NaN,NaN,NaN,NaN,hi,NaN
2,0f018a99663f33afbb7d38aee281fb1afcfd07f9e7acd0...,/content/cv-corpus-25.0-2026-03-09/hi/clips/co...,6f74cd7cea0efbcf45411b70e1eab18ffd4f3e344d332c...,उत्तर कोरिया ने अमेरिका को दी हमले की धमकी,NaN,2,0,NaN,NaN,NaN,NaN,hi,NaN
3,0f018a99663f33afbb7d38aee281fb1afcfd07f9e7acd0...,/content/cv-corpus-25.0-2026-03-09/hi/clips/co...,758c421b84cdaf0ff497c9f62f4d51fc270d4f3c89a013...,अगले कमरे में अनेक रोमन मूर्तियाँ हैं।,NaN,2,0,NaN,NaN,NaN,NaN,hi,NaN
4,0f018a99663f33afbb7d38aee281fb1afcfd07f9e7acd0...,/content/cv-corpus-25.0-2026-03-09/hi/clips/co...,8454c34fb7e19a1d5bd7a859ec4cf1e54237e7d326716f...,तुम ने टॉम को कहाँ भेज दिया?,NaN,2,1,NaN,NaN,NaN,NaN,hi,NaN


In [8]:
train_df.sample(800,random_state=42).shape

(800, 13)

In [9]:
# /content/cv-corpus-25.0-2026-03-09/hi/clips/common_voice_hi_23795238.mp3
train_df['path']

,path
0,/content/cv-corpus-25.0-2026-03-09/hi/clips/co...
1,/content/cv-corpus-25.0-2026-03-09/hi/clips/co...
2,/content/cv-corpus-25.0-2026-03-09/hi/clips/co...
3,/content/cv-corpus-25.0-2026-03-09/hi/clips/co...
4,/content/cv-corpus-25.0-2026-03-09/hi/clips/co...
...,...
4889,/content/cv-corpus-25.0-2026-03-09/hi/clips/co...
4890,/content/cv-corpus-25.0-2026-03-09/hi/clips/co...
4891,/content/cv-corpus-25.0-2026-03-09/hi/clips/co...
4892,/content/cv-corpus-25.0-2026-03-09/hi/clips/co...


## Convert to Hugging Face Dataset

In [10]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df.sample(1200,random_state=42))

## Add audio column

In [11]:
from datasets import Audio

train_dataset = train_dataset.cast_column(
    "path", Audio(sampling_rate=16000)
)

test_dataset = test_dataset.cast_column(
    "path", Audio(sampling_rate=16000)
)

In [12]:
train_dataset[0]['path']['array']

array([ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
       -1.06715824e-07,  2.32571423e-07,  8.11920415e-08], dtype=float32)

### rename `path` → `audio`

In [13]:
train_dataset = train_dataset.rename_column("path", "audio")
test_dataset = test_dataset.rename_column("path", "audio")

In [14]:
from datasets import load_dataset, DatasetDict

common_voice = DatasetDict()

common_voice["train"] = train_dataset
common_voice["test"] = test_dataset

print(common_voice)

DatasetDict({
    train: Dataset({
        features: ['client_id', 'audio', 'sentence_id', 'sentence', 'sentence_domain', 'up_votes', 'down_votes', 'age', 'gender', 'accents', 'variant', 'locale', 'segment'],
        num_rows: 4894
    })
    test: Dataset({
        features: ['client_id', 'audio', 'sentence_id', 'sentence', 'sentence_domain', 'up_votes', 'down_votes', 'age', 'gender', 'accents', 'variant', 'locale', 'segment', '__index_level_0__'],
        num_rows: 1200
    })
})


In [15]:
common_voice = common_voice.remove_columns(['sentence_domain', 'up_votes', 'down_votes', 'age', 'gender', 'accents', 'variant', 'locale', 'segment','sentence_id','client_id'])

In [16]:
common_voice

DatasetDict({
    train: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 4894
    })
    test: Dataset({
        features: ['audio', 'sentence', '__index_level_0__'],
        num_rows: 1200
    })
})

### Load Whisper processor

In [17]:
from transformers import WhisperFeatureExtractor

feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-small")


preprocessor_config.json: 0.00B [00:00, ?B/s]

In [18]:
from transformers import WhisperTokenizer

tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-small", language="Hindi", task="transcribe")


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [19]:
def normalize(text):
    return text.strip()


In [20]:
input_str = normalize(common_voice["train"][0]["sentence"])
labels = tokenizer(input_str).input_ids
decoded_with_special = tokenizer.decode(labels, skip_special_tokens=False)
decoded_str = tokenizer.decode(labels, skip_special_tokens=True)

print(f"Input:                 {input_str}")
print(f"Decoded w/ special:    {decoded_with_special}")
print(f"Decoded w/out special: {decoded_str}")
print(f"Are equal:             {input_str == decoded_str}")


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Input:                 हमने उसका जन्मदिन मनाया।
Decoded w/ special:    <|startoftranscript|><|hi|><|transcribe|><|notimestamps|>हमने उसका जन्मदिन मनाया।<|endoftext|>
Decoded w/out special: हमने उसका जन्मदिन मनाया।
Are equal:             True


In [21]:
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained("openai/whisper-small")

config.json: 0.00B [00:00, ?B/s]

In [22]:
def prepare_dataset(batch):
    audio = batch["audio"]

    batch["input_features"] = processor.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    batch["labels"] = processor.tokenizer(batch["sentence"]).input_ids

    return batch


In [23]:
common_voice = common_voice.map(prepare_dataset, remove_columns=common_voice.column_names["train"])

Map:   0%|          | 0/4894 [00:00<?, ? examples/s]

Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

In [24]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")


model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

In [25]:
model.generation_config.language = "hindi"
model.generation_config.task = "transcribe"

model.generation_config.forced_decoder_ids = None
model.config.suppress_tokens = []


In [26]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any  # Usually a Wav2Vec2Processor or WhisperProcessor
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Separate audio and label features
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        # Pad audio features
        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        # Pad labels
        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        # Replace padding token id's in labels by -100 so they are ignored by loss
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["attention_mask"].ne(1), -100
        )

        # Ensure decoder start token is set
        labels[labels[:, 0] != self.decoder_start_token_id, 0] = self.decoder_start_token_id

        batch["labels"] = labels
        return batch



In [27]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)


In [28]:
import evaluate

metric = evaluate.load("wer")


In [29]:
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}


In [31]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-small-hin",  # change to a repo name of your choice
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,  # increase by 2x for every 2x decrease in batch size
    learning_rate=1e-3,
    warmup_steps=250,
    # max_steps=5000,
    num_train_epochs=4,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=1000,
    eval_steps=1000,
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=True,
)


In [32]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=common_voice["train"],
    eval_dataset=common_voice["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
)


In [ ]:
trainer.train()

Step,Training Loss,Validation Loss,Wer
1000,0.593386,1.093911,86.406235
1224,0.442702,0.909233,79.224623


[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transform

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
kwargs = {
    "dataset_tags": "mozilla-foundation/common_voice_25_0",
    "dataset": "common_voice",  # Use a simpler name for the dataset
    "dataset_args": "hi",
    "language": "hi",
    "model_name": "Whisper Small Hi - Lokesh Gaur",  # a 'pretty' name for your model
    "finetuned_from": "openai/whisper-small",
    "tasks": "automatic-speech-recognition",
}

In [ ]:
trainer.push_to_hub(**kwargs)

In [1]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor

model = WhisperForConditionalGeneration.from_pretrained("lkgaur/whisper-small-hin")
processor = WhisperProcessor.from_pretrained("lkgaur/whisper-small-hin")


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

processor_config.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
from transformers import pipeline
import gradio as gr

# Explicitly set the task for the pipeline
pipe = pipeline("automatic-speech-recognition", model="lkgaur/whisper-small-hin")

def transcribe(audio):
    text = pipe(audio)["text"]
    return text

iface = gr.Interface(
    fn=transcribe,
    inputs=gr.Audio(sources=["microphone"], type="filepath"),
    outputs="text",
    title="Whisper Small Hindi",
    description="Realtime demo for Hindi speech recognition using a fine-tuned Whisper small model.",
)

iface.launch()